# 03. Data Pipeline Integration

This notebook demonstrates the complete data pipeline with all components integrated.

In [10]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Reset logging configuration (helpful for Jupyter notebook caching issues)
from src.utils.logger import reset_logging_config, setup_logger_safe
reset_logging_config()

# Force refresh of all cached src modules (for Jupyter kernel caching issues)
import importlib
modules_to_reload = [
    'src.data.data_loader',
    'src.data.cache_manager', 
    'src.data.data_versioning',
    'src.utils.config_manager',
    'src.utils.data_validation'
]

reloaded_count = 0
for module_name in modules_to_reload:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
        reloaded_count += 1

if reloaded_count > 0:
    print(f"🔄 Reloaded {reloaded_count} cached modules")

# Import our modules
from src.data.data_loader import PayloadByteDataLoader, create_pytorch_dataloaders
from src.data.cache_manager import CacheManager
from src.data.data_versioning import DataVersionManager
from src.utils.config_manager import get_config_manager
from src.utils.data_validation import DataValidator

logger = setup_logger_safe('data_pipeline_demo')
print("✅ All modules imported successfully!")

🔄 Reloaded 5 cached modules
✅ All modules imported successfully!


## 1. Configuration Management

In [21]:
# Load configuration
config_mgr = get_config_manager()

# Display data configuration
data_config = config_mgr.get_data_config()
print("Data Configuration:")
print(f"  Raw data path: {data_config['data']['raw_data_path']}")
print(f"  Chunk size: {data_config['data']['chunk_size']}")
print(f"  Workers: {data_config['data']['n_workers']}")
print(f"\nSplit ratios:")
for split, ratio in data_config['data']['split_ratios'].items():
    print(f"  {split}: {ratio}")

Data Configuration:
  Raw data path: data/raw/payload_byte
  Chunk size: 10000
  Workers: 4

Split ratios:
  train: 0.7
  validation: 0.15
  test: 0.15


## 2. Data Validation

In [22]:
# Validate datasets
validator = DataValidator()
data_path = project_root / data_config['data']['raw_data_path']

# Run validation
validation_results = validator.validate_dataset(data_path)
validator.print_summary(validation_results)

2025-06-21 12:33:58 - data_validation - INFO - Validating cic_ids2017_sample.csv...
2025-06-21 12:33:58 - data_validation - INFO - Validating unsw_nb15_sample.csv...

DATA VALIDATION SUMMARY
Total Files: 2
Valid Files: 2
Invalid Files: 0
Validation Rate: 100.0%



## 3. Data Versioning

In [23]:
# Force refresh of cached modules (for Jupyter kernel caching issues)
import importlib
if 'src.data.data_versioning' in sys.modules:
    importlib.reload(sys.modules['src.data.data_versioning'])
    print("🔄 Reloaded data_versioning module")

# Re-import after reload
from src.data.data_versioning import DataVersionManager

# Initialize version manager
version_mgr = DataVersionManager()

# Create version for our datasets
print("Creating data versions...")
for csv_file in data_path.glob('*.csv'):
    if 'sample' in csv_file.name:
        try:
            version = version_mgr.create_version(
                data_path=csv_file,
                description=f"Pipeline demo version of {csv_file.name}",
                tags=['demo', 'pipeline']
            )
            print(f"✅ Version ready: {version.version_id} for {csv_file.name}")
        except Exception as e:
            print(f"❌ Error creating version for {csv_file.name}: {e}")
            # Enhanced error info for debugging
            import traceback
            traceback.print_exc()

# List all versions
print("\nAll data versions:")
try:
    versions = version_mgr.list_versions()
    if versions:
        for v in versions:
            print(f"  {v.version_id}: {v.metadata['description']}")
    else:
        print("  No versions found")
        
    print(f"\nTotal versions: {len(versions)}")
except Exception as e:
    print(f"❌ Error listing versions: {e}")
    import traceback
    traceback.print_exc()

🔄 Reloaded data_versioning module
2025-06-21 12:34:01 - data_versioning - INFO - Data version manager initialized: /home/ubuntu/analyst/data/versions
Creating data versions...
2025-06-21 12:34:01 - data_versioning - WARNING - Data version already exists: v_20250621_122215_a21b30400c8b5d11
✅ Version ready: v_20250621_122215_a21b30400c8b5d11 for cic_ids2017_sample.csv
2025-06-21 12:34:01 - data_versioning - WARNING - Data version already exists: v_20250621_122714_c6cf6bdefc5aee10
✅ Version ready: v_20250621_122714_c6cf6bdefc5aee10 for unsw_nb15_sample.csv

All data versions:
  v_20250621_122714_c6cf6bdefc5aee10: Pipeline demo version of unsw_nb15_sample.csv
  v_20250621_122215_a21b30400c8b5d11: Pipeline demo version of cic_ids2017_sample.csv

Total versions: 2


## 4. Data Loading with Caching

In [24]:
# Initialize cache manager
cache_mgr = CacheManager()
print(f"Cache enabled: {cache_mgr.enabled}")
print(f"Cache format: {cache_mgr.format}")
print(f"Cache TTL: {cache_mgr.ttl_hours} hours")

# Initialize data loader
loader = PayloadByteDataLoader(
    data_path=data_path,
    batch_size=config_mgr.get('data_config', 'pipeline.batch_size', 32),
    chunk_size=data_config['data']['chunk_size'],
    n_workers=data_config['data']['n_workers']
)

2025-06-21 12:34:07 - cache_manager - INFO - Cache manager initialized: data/cache
Cache enabled: True
Cache format: hdf5
Cache TTL: 24 hours
2025-06-21 12:34:07 - data_loader - INFO - Found 2 data files


In [25]:
# Load data with caching
import time

# First load (will be cached)
start_time = time.time()
data = loader.load_raw_packets(nrows=1000)  # Load subset for demo
first_load_time = time.time() - start_time
print(f"First load time: {first_load_time:.2f}s")
print(f"Loaded {len(data)} rows")

# Cache the data
cache_key_path = list(data_path.glob('*.csv'))[0]
cache_mgr.save_to_cache(data, cache_key_path, {'nrows': 1000})

# Second load (from cache)
start_time = time.time()
cached_data = cache_mgr.load_from_cache(cache_key_path, {'nrows': 1000})
cache_load_time = time.time() - start_time

if cached_data is not None:
    print(f"\nCache load time: {cache_load_time:.2f}s")
    print(f"Speedup: {first_load_time / cache_load_time:.1f}x")
else:
    print("Cache miss")

2025-06-21 12:34:10 - data_loader - INFO - Loading data from 2 files...
2025-06-21 12:34:10 - data_loader - INFO - Loaded 2,000 total packets
First load time: 0.22s
Loaded 2000 rows
2025-06-21 12:34:11 - cache_manager - INFO - Saved to cache: cf6e0254ffb5c9880aeb5df75596b5a2 (2000 rows, 2.73 MB in 0.18s)
2025-06-21 12:34:11 - cache_manager - INFO - Loaded from cache: cf6e0254ffb5c9880aeb5df75596b5a2 (2000 rows in 0.08s)

Cache load time: 0.08s
Speedup: 2.7x


## 5. Train/Validation/Test Splitting

In [26]:
# Create splits
train_df, val_df, test_df = loader.create_train_val_test_splits(
    data=data,
    train_ratio=data_config['data']['split_ratios']['train'],
    val_ratio=data_config['data']['split_ratios']['validation'],
    test_ratio=data_config['data']['split_ratios']['test'],
    stratify=data_config['data']['stratify'],
    random_state=data_config['data']['random_state']
)

# Display split statistics
stats = loader.get_data_stats()
print("\nData Split Statistics:")
print(f"  Train samples: {stats['n_train']}")
print(f"  Val samples: {stats['n_val']}")
print(f"  Test samples: {stats['n_test']}")

# Check label distribution
if 'train_labels' in stats:
    print("\nTrain set label distribution:")
    for label, count in stats['train_labels'].items():
        print(f"  Label {label}: {count} samples")

2025-06-21 12:34:15 - data_loader - INFO - Creating splits: train=0.7, val=0.15, test=0.15
2025-06-21 12:34:15 - data_loader - INFO - Split sizes - Train: 1,400, Val: 300, Test: 300

Data Split Statistics:
  Train samples: 1400
  Val samples: 300
  Test samples: 300

Train set label distribution:
  Label 0: 249 samples
  Label 2: 243 samples
  Label 5: 237 samples
  Label 3: 232 samples
  Label 4: 222 samples
  Label 1: 217 samples


## 6. Batch Generation

In [27]:
# Test batch generator
print("Testing batch generator:")
batch_gen = loader.get_batch_generator('train', shuffle=True)

for i, batch in enumerate(batch_gen):
    print(f"  Batch {i}: shape={batch.shape}")
    if i >= 4:  # Show first 5 batches
        break

# Visualize a batch
first_batch = next(loader.get_batch_generator('train'))
print(f"\nFirst batch details:")
print(f"  Shape: {first_batch.shape}")
print(f"  Columns: {len(first_batch.columns)}")
print(f"  Memory usage: {first_batch.memory_usage().sum() / 1024:.2f} KB")

Testing batch generator:
  Batch 0: shape=(32, 1505)
  Batch 1: shape=(32, 1505)
  Batch 2: shape=(32, 1505)
  Batch 3: shape=(32, 1505)
  Batch 4: shape=(32, 1505)

First batch details:
  Shape: (32, 1505)
  Columns: 1505
  Memory usage: 376.50 KB


## 7. PyTorch Integration

In [28]:
# Force refresh of data_loader module (fixes mixed data type handling)
import importlib
if 'src.data.data_loader' in sys.modules:
    importlib.reload(sys.modules['src.data.data_loader'])
    print("🔄 Reloaded data_loader module")

# Re-import after reload
from src.data.data_loader import create_pytorch_dataloaders

# Create PyTorch DataLoaders
dataloaders = create_pytorch_dataloaders(
    loader,
    batch_size=32,
    num_workers=2,
    pin_memory=False  # Set to True if using GPU
)

print("PyTorch DataLoaders created:")
for name, dataloader in dataloaders.items():
    print(f"  {name}: {len(dataloader)} batches")

# Test PyTorch DataLoader
train_loader = dataloaders['train']
batch_x, batch_y = next(iter(train_loader))
print(f"\nPyTorch batch:")
print(f"  Features shape: {batch_x.shape}")
print(f"  Labels shape: {batch_y.shape}")
print(f"  Features dtype: {batch_x.dtype}")
print(f"  Labels dtype: {batch_y.dtype}")

print(f"\n✅ PyTorch integration working! Using {batch_x.shape[1]} numeric features.")

🔄 Reloaded data_loader module
2025-06-21 12:34:22 - data_loader - WARNING - Skipping non-numeric column: src_ip (dtype: object)
2025-06-21 12:34:22 - data_loader - WARNING - Skipping non-numeric column: dst_ip (dtype: object)
2025-06-21 12:34:22 - data_loader - INFO - Using 1502 numeric features out of 1504 total columns
2025-06-21 12:34:22 - data_loader - WARNING - Skipping non-numeric column: src_ip (dtype: object)
2025-06-21 12:34:22 - data_loader - WARNING - Skipping non-numeric column: dst_ip (dtype: object)
2025-06-21 12:34:22 - data_loader - INFO - Using 1502 numeric features out of 1504 total columns
2025-06-21 12:34:22 - data_loader - WARNING - Skipping non-numeric column: src_ip (dtype: object)
2025-06-21 12:34:22 - data_loader - WARNING - Skipping non-numeric column: dst_ip (dtype: object)
2025-06-21 12:34:22 - data_loader - INFO - Using 1502 numeric features out of 1504 total columns
PyTorch DataLoaders created:
  train: 44 batches
  val: 10 batches
  test: 10 batches

PyTo

## 8. Cache Statistics

In [29]:
# Display cache statistics
cache_stats = cache_mgr.get_cache_stats()
print("\nCache Statistics:")
for key, value in cache_stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")


Cache Statistics:
  total_entries: 1
  total_size_mb: 2.73
  format: hdf5
  compression: gzip
  ttl_hours: 24
  total_accesses: 1
  avg_accesses_per_entry: 1.00
  oldest_entry_age_hours: 0.01
  newest_entry_age_hours: 0.01


## 9. Pipeline Performance Metrics

In [30]:
# Measure pipeline performance
import time

performance_metrics = {}

# Data loading speed
start = time.time()
_ = loader.load_raw_packets(nrows=100)
performance_metrics['load_time_per_100_rows'] = time.time() - start

# Batch generation speed
start = time.time()
batches = list(loader.get_batch_generator('train'))
performance_metrics['batch_generation_time'] = time.time() - start
performance_metrics['batches_per_second'] = len(batches) / performance_metrics['batch_generation_time']

# Memory usage
import psutil
process = psutil.Process()
performance_metrics['memory_usage_mb'] = process.memory_info().rss / 1024 / 1024

print("Pipeline Performance Metrics:")
for metric, value in performance_metrics.items():
    print(f"  {metric}: {value:.2f}")

2025-06-21 12:34:35 - data_loader - INFO - Loading data from 2 files...
2025-06-21 12:34:35 - data_loader - INFO - Loaded 200 total packets
Pipeline Performance Metrics:
  load_time_per_100_rows: 0.10
  batch_generation_time: 0.03
  batches_per_second: 1666.75
  memory_usage_mb: 944.09


## Summary

The data pipeline successfully integrates:

1. **Configuration Management**: Centralized YAML-based configuration
2. **Data Validation**: Comprehensive validation of data format and integrity
3. **Data Versioning**: Track dataset versions and lineage
4. **Efficient Loading**: Chunked reading with multi-threading support
5. **Caching**: HDF5-based caching for fast data access
6. **Train/Val/Test Splitting**: Stratified splitting with configurable ratios
7. **Batch Generation**: Memory-efficient batch generation
8. **PyTorch Integration**: Ready-to-use DataLoaders for model training

The pipeline is modular, configurable, and optimized for performance.